In [66]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [67]:
annual_value_adjusted = pd.read_csv("data/outputs/annual_ca_port",
                                    index_col =0, parse_dates = [1])
quarterly_value_adjusted = pd.read_csv('data/outputs/quarter_ca_port',
                                    index_col =0, parse_dates = [1])
band_value_adjusted = pd.read_csv("data/outputs/quarter_band_ca_port",
                                    index_col =0, parse_dates = [1])
asset_data = pd.read_csv("data/processed/daily_returns.csv",
                         parse_dates = [0])


In [74]:
annual_weights = pd.read_csv("data/outputs/annually_rebalanced_weights", 
                             index_col = 0, parse_dates = [1])
quarterly_weights = pd.read_csv("data/outputs/quarterly_rebalanced_weights", 
                                index_col = 0, parse_dates = [1])
band_weights = pd.read_csv("data/outputs/quarterly_rebalanced_byband_weights", 
                           index_col = 0, parse_dates = [1])
daily_returns = pd.read_csv("data/processed/daily_returns.csv", parse_dates = [0])

In [106]:
asset_data = daily_returns[daily_returns['Date'].isin(annual_value_adjusted['Date'])].reset_index().drop(columns=['index'])

In [115]:
def daily_asset_contributions(weights_df, asset_df):

    starting_weights = weights_df[weights_df['weight_type'] == 'starting']

    start_dates = weights_df.loc[weights_df['weight_type'] == 'starting', 'Date']

    end_dates = weights_df.loc[weights_df['weight_type'] == 'ending', 'Date']

    returns_list = []

    for row in range(len(start_dates)):

        start = start_dates.iloc[row]
        
        end = end_dates.iloc[row]

        period_data = asset_df[asset_df['Date'].between(start, end)].reset_index()

        weights = starting_weights[starting_weights['Date'] == start].drop(columns = ['Date','weight_type'])

        for row in range(len(period_data)):

            returns_by_asset = weights.iloc[0] * period_data.iloc[row]

            returns_by_asset['Date'] = period_data['Date'].iloc[row]

            returns_list.append(returns_by_asset)

    return pd.DataFrame(returns_list).set_index('Date')


In [116]:
daily_asset_contributions(annual_weights, asset_data)

,AGG,EEM,EFA,GLD,IWM,LQD,MTUM,QQQ,QUAL,SPY,TLT,USMV,VLUE,VNQ,index
Date,,,,,,,,,,,,,,,
2022-01-03,-0.001026,0.000000,0.000972,-0.000000,7.330010e-19,-1.831623e-18,0.0,0.000481,-0.000134,0.000869,-0.003937,-0.000927,0.000000,-0.001177,NaN
2022-01-04,-0.000013,-0.000000,0.000852,0.000000,-8.572949e-20,1.443801e-19,-0.0,-0.000649,-0.000059,-0.000050,-0.000624,-0.000504,0.000000,-0.000182,NaN
2022-01-05,-0.000463,-0.000000,-0.001318,-0.000000,-1.924258e-18,-1.010114e-18,-0.0,-0.001536,-0.001203,-0.002880,-0.000814,-0.001797,-0.000000,-0.004293,NaN
2022-01-06,-0.000159,0.000000,-0.000760,-0.000000,3.241178e-19,-2.243575e-19,0.0,-0.000035,0.000046,-0.000141,0.000388,-0.000625,0.000000,0.000134,NaN
2022-01-07,-0.000439,0.000000,0.000439,0.000000,-6.575373e-19,-7.399853e-19,-0.0,-0.000542,-0.000470,-0.000593,-0.001078,-0.000552,0.000000,-0.000993,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-07-06,0.000076,0.004269,0.000000,0.001587,0.000000e+00,4.141952e-05,0.0,0.000717,0.000000,0.001309,-0.000000,0.000000,0.000801,-0.000000,NaN
2026-07-07,-0.000684,-0.004107,-0.000000,-0.001821,-0.000000e+00,-1.090459e-03,-0.0,-0.000926,-0.000000,-0.000713,-0.000000,0.000235,-0.001647,0.000000,NaN
2026-07-08,-0.000260,0.001164,-0.000000,-0.001208,-0.000000e+00,-2.919898e-04,0.0,0.000142,-0.000000,-0.000463,-0.000000,-0.000356,-0.000321,-0.000000,NaN


In [118]:

annual_portfolio_asset_contributions = daily_asset_contributions(annual_weights, asset_data)
quarterly_portfolio_asset_contributions = daily_asset_contributions(quarterly_weights, asset_data)
banded_portfolio_asset_contributions = daily_asset_contributions(band_weights, asset_data)

In [ ]:
def asset_contributions_to_returns(portfolio_df, asset_contribution_df):
    weights_sums = []
    total_return = portfolio_df['portfolio_return'].sum()

    for col in asset_contribution_df.columns:
        name = asset_contribution_df[col].name
        col_sum = asset_contribution_df[col].sum()
        percent_cont = col_sum / total_return
        weights_sums.append({
            'asset' : name, 
            'total_return' : col_sum,
            'percent_contribution' : (f'{percent_cont * 100 :.3f} %')
        })

    return pd.DataFrame(weights_sums)

asset_contribution_table_band = asset_contributions_to_returns(band_value_adjusted, banded_portfolio_asset_contributions)